In [1]:
! pip install -q google-generativeai

In [2]:
import google.generativeai as genai
from google.colab import userdata

api_key=userdata.get("TitanicAPI")
genai.configure(api_key=api_key)

In [21]:
# Initialize the model
model=genai.GenerativeModel("gemini-1.5-flash")

In [4]:
import pandas as pd
import seaborn as sns

df=sns.load_dataset("titanic")
df.head(3)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


Step-1 = SQL

In [5]:
prompt = f'''look the variable {df}, analyse the dataset and answer the questions that are given below.
Explain what the differences in survival rate between male and female passengers suggest about evacuation priorities on the Titanic.
Interpret how passenger class influenced survival chances. Does being in 1st, 2nd, or 3rd class significantly impact the survival probability?
Compare the average age of survivors vs. non-survivors. What does this reveal about age as a factor in survival?
For all three quewtions, give me SQL queries to support this'''

response=model.generate_content(prompt)
print(response.text)

The provided data is a Pandas DataFrame, not a SQL database.  Therefore, I cannot provide SQL queries to directly analyze it. To answer your questions, I'll use Python with Pandas and provide the equivalent SQL-like logic in comments.


```python
import pandas as pd
import numpy as np

data = """survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,3,male,22.0,1,0,7.2500,S,Third,man,True,,Southampton,no,False
1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,,Southampton,no,True
0,2,male,27.0,0,0,13.0000,S,Second,man,True,,Southampton,no,True
1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
0,3,female,,1,2,23.4500,S,Third,woman,False,,Southampton,no,False
1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,ye

In [6]:
import duckdb
import pandas as pd

query1 = """SELECT
    pclass,
    SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) AS survivors,
    COUNT(*) AS total_passengers,
    CAST(SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) AS survival_rate
FROM
    df
GROUP BY
    pclass
ORDER BY
    pclass;"""

result = duckdb.query(query1).to_df()

print(result)


   pclass  survivors  total_passengers  survival_rate
0       1      136.0               216      62.962963
1       2       87.0               184      47.282608
2       3      119.0               491      24.236252


In [7]:
import duckdb
import pandas as pd

query1 = """SELECT
    pclass,
    SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) AS survivors,
    COUNT(*) AS total_passengers,
    CAST(SUM(CASE WHEN survived = 1 THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(*) AS survival_rate
FROM
    df
GROUP BY
    pclass
ORDER BY
    pclass;"""

result = duckdb.query(query1).to_df()

print(result)


   pclass  survivors  total_passengers  survival_rate
0       1      136.0               216      62.962963
1       2       87.0               184      47.282608
2       3      119.0               491      24.236252


In [8]:
import duckdb
import pandas as pd

query = """SELECT
    survived,
    AVG(age) AS average_age
FROM
    df
GROUP BY
    survived;"""

result = duckdb.query(query).to_df()

print(result)


   survived  average_age
0         0    30.626179
1         1    28.343690


Step-2 = Feature Engineering

In [9]:
prompt1 = f'''look the variable {df}, analyse the dataset and perform some feature engineering using pandas and give me some new columns in the dataset as well so
 that I can create models from this.
'''

response=model.generate_content(prompt1)
print(response.text)

This dataset appears to be from the Titanic passenger manifest.  Let's perform some feature engineering using pandas to create new columns suitable for machine learning model building.  The goal is to create features that might be predictive of survival.

```python
import pandas as pd
import numpy as np

# Sample data (replace with your actual data loading)
data = {'survived': [0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0], 
        'pclass': [3, 1, 3, 1, 3, 2, 3, 3, 1, 1, 2], 
        'sex': ['male', 'female', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'male'], 
        'age': [22.0, 38.0, 26.0, 35.0, 35.0, np.nan, 24.0, 30.0, 28.0, 26.0, 27.0], 
        'sibsp': [1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0], 
        'parch': [0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 1], 
        'fare': [7.25, 71.2833, 7.925, 53.1, 8.05, 13.0, 11.2417, 7.8542, 80.0, 30.0, 13.0],
        'embarked': ['S', 'C', 'S', 'S', 'S', 'Q', 'Q', 'S', 'C', 'C', 'Q'],
        'class': ['Third', 'First', 'Third', 'First'

In [10]:
# This dataset appears to be from the Titanic. Let's perform some feature engineering using pandas to create new columns suitable for model building.

# python
import pandas as pd
import numpy as np


# Feature Engineering

# 1. Family Size:
df['family_size'] = df['sibsp'] + df['parch'] + 1

# 2. Is Alone (already present, but can be recalculated for consistency):
df['is_alone'] = (df['family_size'] == 1).astype(int)


# 3. Age Group:  Binning ages into categories. Adjust bins as needed.
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 60, 100], labels=['child', 'adult', 'senior'], right=False)
# Add 'unknown' to categories before filling NaN
df['age_group'] = df['age_group'].cat.add_categories('unknown').fillna('unknown')


# 4. Fare per Person:
df['fare_per_person'] = df['fare'] / df['family_size']

#5. Embarked Numerical Encoding.  This is one way. You might prefer one-hot encoding.
embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
df['embarked_num'] = df['embarked'].map(embarked_mapping)

# 6. Handle Missing Age (more sophisticated methods exist):
# Here we fill NaN with the median age based on passenger class.
df['age'] = df.groupby('pclass')['age'].transform(lambda x: x.fillna(x.median()))


#7. Sex Numerical encoding
df['sex'] = df['sex'].map({'male':0, 'female':1})

# 8. Convert 'alive' to numerical (0/1):
df['alive_num'] = df['alive'].map({'no': 0, 'yes': 1})


# Display the updated DataFrame
print(df)


# This code adds several new features:  `family_size`, `is_alone`, `age_group`, `fare_per_person`, `embarked_num`, and `alive_num`.  It also handles missing age values and converts categorical features to numerical representations.  Remember to choose appropriate imputation and encoding strategies based on your modeling technique and the characteristics of your dataset.  One-hot encoding for categorical variables might be preferable to numerical encoding in some cases (especially for tree-based models).  Consider exploring other feature engineering possibilities depending on the specific insights you are seeking from the Titanic dataset.

     survived  pclass  sex   age  sibsp  parch     fare embarked   class  \
0           0       3    0  22.0      1      0   7.2500        S   Third   
1           1       1    1  38.0      1      0  71.2833        C   First   
2           1       3    1  26.0      0      0   7.9250        S   Third   
3           1       1    1  35.0      1      0  53.1000        S   First   
4           0       3    0  35.0      0      0   8.0500        S   Third   
..        ...     ...  ...   ...    ...    ...      ...      ...     ...   
886         0       2    0  27.0      0      0  13.0000        S  Second   
887         1       1    1  19.0      0      0  30.0000        S   First   
888         0       3    1  24.0      1      2  23.4500        S   Third   
889         1       1    0  26.0      0      0  30.0000        C   First   
890         0       3    0  32.0      0      0   7.7500        Q   Third   

       who  ...  deck  embark_town alive  alone  family_size  is_alone  \
0      man  .

Step-3 : ML Models Recommendation and hyper tuning

In [11]:
prompt1 = f'''look the variable {df}, analyse the dataset and recommend me some ML models that can be created on it and what kind of hyperparameters can be used to upgrade the ML models.
'''

response=model.generate_content(prompt1)
print(response.text)

This dataset appears to be from the Titanic passenger manifest, commonly used for machine learning exercises.  The goal is usually to predict passenger survival (`survived`). Let's analyze it and suggest suitable ML models and hyperparameters.


**Data Analysis and Preprocessing:**

Before model selection, several preprocessing steps are crucial:

1. **Handling Missing Values:** The `age` and `deck` columns likely have missing values.  We can:
    * **Age:** Impute missing ages using the mean, median, or a more sophisticated method like K-Nearest Neighbors (KNN) imputation.
    * **Deck:**  Either impute with a new category ("Unknown") or drop the column if it doesn't significantly contribute to prediction.

2. **Encoding Categorical Features:**  Convert categorical features (like `sex`, `embarked`, `class`, `who`, `embark_town`, `deck` (if kept)) into numerical representations using one-hot encoding or label encoding.

3. **Feature Engineering:** The dataset already includes some engi

Step-4- Training Models

In [12]:
prompt1 = f'''look the variable {df}, analyse the dataset and train some ML models and then Evaluate these results for mulitple models.
'''

response=model.generate_content(prompt1)
print(response.text)

This code performs exploratory data analysis (EDA), preprocesses the data, trains several machine learning models to predict passenger survival, and evaluates their performance.

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

# Load the dataset (assuming it's in a CSV file named 'titanic.csv')
df = pd.read_csv('titanic.csv') # Replace 'titanic.csv' with your file name if different


# Data Preprocessing and Feature Engineering
# (This section is more comp

In [16]:
# This code performs exploratory data analysis (EDA), preprocesses the data, trains several machine learning models to predict passenger survival, and evaluates their performance using various metrics.

# python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score


# Use the existing DataFrame 'df' loaded from seaborn
# Load the dataset (assuming it's a CSV file named 'titanic.csv')
# try:
# df = pd.read_csv('titanic.csv')

# except FileNotFoundError:
#print("Error: titanic.csv not found. Please make sure the file is in the same directory.")
#exit()


# One-hot encode categorical features
# categorical_cols = ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive']
# df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features (X) and target (y)
X = df.drop('survived', axis=1)
y = df['survived']

# Drop original categorical columns after one-hot encoding
# X = X.drop(columns=categorical_cols, errors='ignore')


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Create a preprocessing pipeline (handles numerical and categorical features differently)

# Identify numerical and categorical features based on the current state of X
numerical_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=[np.number, np.bool_]).columns # Exclude boolean columns


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)])

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(random_state=42)
}


# Train and evaluate models
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)

    cv_scores = cross_val_score(pipeline, X, y, cv=5) # 5-fold cross-validation

    results[name] = {
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': conf_matrix,
        'cv_scores': cv_scores,
        'cv_mean': np.mean(cv_scores),
        'cv_std': np.std(cv_scores)
    }

# Print results
for name, result in results.items():
    print(f"Model: {name}\n")
    print(f"Accuracy: {result['accuracy']:.4f}\n")
    print(f"Classification Report:\n{result['report']}\n")
    print(f"Confusion Matrix:\n{result['confusion_matrix']}\n")
    print(f"Cross-Validation Scores: {result['cv_scores']}\n")
    print(f"Cross-Validation Mean: {result['cv_mean']:.4f}\n")
    print(f"Cross-Validation Standard Deviation: {result['cv_std']:.4f}\n")
    print("-" * 20)

# ```

#Remember to install the necessary libraries: `pandas`, `numpy`, `scikit-learn`.  You can install them using pip:  `pip install pandas numpy scikit-learn`

#This improved code handles missing values more robustly, uses pipelines for better organization and reproducibility, and includes cross-validation for a more reliable performance estimate.  The output shows accuracy, classification reports, confusion matrices, and cross-validation results for each model, allowing for a more thorough comparison.  Remember that the best model will depend on your specific priorities (e.g., precision, recall, F1-score).

Model: Logistic Regression

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179


Confusion Matrix:
[[105   0]
 [  0  74]]

Cross-Validation Scores: [1. 1. 1. 1. 1.]

Cross-Validation Mean: 1.0000

Cross-Validation Standard Deviation: 0.0000

--------------------
Model: Random Forest

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179


Confusion Matrix:
[[105   0]
 

In [23]:
genai_model=genai.GenerativeModel("gemini-1.5-flash")

In [25]:
prompt1 = f'''Based on {df}, implement some Feature Engineering, train some ML models, evaluate the results and and then generate me a complete report on the best model that has
been made from this analysis or dataset.
Give me a detailed report on this in step by step manner and include all the points in this.
'''

response=genai_model.generate_content(prompt1)
print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 2149.56ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 3099.82ms


## Titanic Survival Prediction: A Machine Learning Approach

This report details a machine learning project aimed at predicting passenger survival on the Titanic, using the provided dataset.  The process involves feature engineering, model training, evaluation, and selection of the best performing model.

**Step 1: Data Preparation and Feature Engineering**

The provided dataset contains various features related to passengers. Before model training, several steps were undertaken:

* **Handling Missing Values:**  While the dataset appears largely complete from the preview, any missing values (if present in the full dataset) would need to be addressed. Strategies could include imputation (filling with mean, median, or mode) or removal of rows with missing data.  The choice depends on the extent of missingness and the nature of the variable.

* **Feature Scaling:**  Features with significantly different scales (e.g., 'age' and 'fare') might need scaling to prevent features with larger val